In [18]:
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [19]:
with h5py.File('processed_physics_data.h5', 'r') as f:
    X_train_raw = np.expand_dims(f['X_train'][:], -1)
    y_train_raw = f['y_train'][:]
    X_val = np.expand_dims(f['X_val'][:], -1)
    y_val = f['y_val'][:]
    
print(f"Original training size: {X_train_raw.shape[0]}")

Original training size: 87778


In [20]:
X_train_flipped_h = np.flip(X_train_raw, axis=2) 
X_train_flipped_v = np.flip(X_train_raw, axis=1) 

X_train = np.concatenate([X_train_raw, X_train_flipped_h, X_train_flipped_v], axis=0)
y_train = np.concatenate([y_train_raw, y_train_raw, y_train_raw], axis=0)

print(f"New amplified training size: {X_train.shape[0]}")
del X_train_raw, X_train_flipped_h, X_train_flipped_v

New amplified training size: 263334


In [21]:
# Add the X/Y grid to the new massive training set
num_train, h, w, _ = X_train.shape
grid_x, grid_y = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
grid_x = np.tile(grid_x[np.newaxis, ..., np.newaxis], (num_train, 1, 1, 1))
grid_y = np.tile(grid_y[np.newaxis, ..., np.newaxis], (num_train, 1, 1, 1))

X_train_cc = np.concatenate([X_train, grid_x, grid_y], axis=-1)
del X_train, grid_x, grid_y

In [5]:
num_val = X_val.shape[0]
grid_x_v, grid_y_v = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
grid_x_v = np.tile(grid_x_v[np.newaxis, ..., np.newaxis], (num_val, 1, 1, 1))
grid_y_v = np.tile(grid_y_v[np.newaxis, ..., np.newaxis], (num_val, 1, 1, 1))

X_val_cc = np.concatenate([X_val, grid_x_v, grid_y_v], axis=-1)

del X_val, grid_x_v, grid_y_v

In [23]:
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [24]:
# with tf.device('/CPU:0'):
#     train_ds = tf.data.Dataset.from_tensor_slices((X_train_cc, y_train))
#     val_ds = tf.data.Dataset.from_tensor_slices((X_val_cc, y_val))
# train_ds = train_ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)
# val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [25]:
def resnet_block(x, filters, kernel_size=3, stride=1):
    shortcut = x
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, kernel_size, strides=stride, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(filters, kernel_size, strides=1, padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

In [26]:
input_shape = (X_train_cc.shape[1], X_train_cc.shape[2], X_train_cc.shape[3])
inputs = Input(shape=input_shape)

x = Conv2D(32, 3, padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

In [27]:
x = resnet_block(x, 32)
x = MaxPooling2D(2)(x)

x = resnet_block(x, 64)
x = MaxPooling2D(2)(x)

x = resnet_block(x, 128)

In [28]:
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs, name="ResNet_CoordConv")
model.summary()

Model: "ResNet_CoordConv"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 24, 36, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 24, 36,    │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 36,    │        128 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 24, 36,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 24, 36,    │      9,248 │ activation_7[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 36,    │        128 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 24, 36,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 24, 36,    │      9,248 │ activation_8[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 36,    │        128 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 24, 36,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ activation_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 24, 36,    │          0 │ add_3[0][0]       │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 12, 18,    │          0 │ activation_9[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 12, 18,    │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 18,    │        256 │ conv2d_13[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 12, 18,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 12, 18,    │     36,928 │ activation_10[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 12, 18,    │      2,112 │ max_pooling2d_2[

 Total params: 317,697 (1.21 MB)

 Trainable params: 316,353 (1.21 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['auc']
)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max')
]

In [14]:
history = model.fit(
    train_ds, 
    validation_data=val_ds,
    epochs`=200,
    callbacks=callbacks
)

Epoch 1/200


2026-07-13 14:46:54.268277: I external/local_xla/xla/service/service.cc:168] XLA service 0x651b5cfcb1a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-13 14:46:54.268299: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GT 1030, Compute Capability 6.1
2026-07-13 14:46:54.323685: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-07-13 14:46:54.672836: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
2026-07-13 14:47:07.077558: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_11', 4 bytes spill stores, 12 bytes spill loads



   5/8230 ━━━━━━━━━━━━━━━━━━━━ 5:21 39ms/step - auc: 0.5380 - loss: 0.3527 

I0000 00:00:1783921627.425783 3428081 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


8229/8230 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - auc: 0.8341 - loss: 0.2398

2026-07-13 14:52:32.461399: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_11', 4 bytes spill stores, 12 bytes spill loads



8230/8230 ━━━━━━━━━━━━━━━━━━━━ 353s 41ms/step - auc: 0.8444 - loss: 0.2318 - val_auc: 0.7337 - val_loss: 0.2908 - learning_rate: 0.0010
Epoch 2/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8570 - loss: 0.2249 - val_auc: 0.7479 - val_loss: 0.2869 - learning_rate: 0.0010
Epoch 3/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8625 - loss: 0.2221 - val_auc: 0.7352 - val_loss: 0.2917 - learning_rate: 0.0010
Epoch 4/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8652 - loss: 0.2205 - val_auc: 0.7458 - val_loss: 0.2887 - learning_rate: 0.0010
Epoch 5/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 316s 38ms/step - auc: 0.8668 - loss: 0.2198 - val_auc: 0.7717 - val_loss: 0.2634 - learning_rate: 0.0010
Epoch 6/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 321s 39ms/step - auc: 0.8685 - loss: 0.2187 - val_auc: 0.7693 - val_loss: 0.2641 - learning_rate: 0.0010
Epoch 7/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 310s 38ms/step - auc: 0.8689 - loss: 0.2185 - val_auc: 0.7595 - val_loss: 0.2790 

8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8797 - loss: 0.2066 - val_auc: 0.8347 - val_loss: 0.2348 - learning_rate: 7.8125e-06
Epoch 46/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 321s 39ms/step - auc: 0.8800 - loss: 0.2063 - val_auc: 0.8341 - val_loss: 0.2352 - learning_rate: 7.8125e-06
Epoch 47/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8799 - loss: 0.2065 - val_auc: 0.8347 - val_loss: 0.2350 - learning_rate: 7.8125e-06
Epoch 48/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8795 - loss: 0.2067 - val_auc: 0.8358 - val_loss: 0.2343 - learning_rate: 7.8125e-06
Epoch 49/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8801 - loss: 0.2063 - val_auc: 0.8362 - val_loss: 0.2345 - learning_rate: 7.8125e-06
Epoch 50/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8799 - loss: 0.2061 - val_auc: 0.8362 - val_loss: 0.2341 - learning_rate: 7.8125e-06
Epoch 51/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 322s 39ms/step - auc: 0.8805 - loss: 0.2061 - val_a

In [29]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.9), # Focuses on tricky ring boundaries
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

In [30]:
neg_count = len(y_train) - sum(y_train)
pos_count = sum(y_train)
total = len(y_train)

class_weights = {
    0: (1 / neg_count) * (total / 2.0),
    1: (1 / pos_count) * (total / 2.0)
}

In [32]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/200


2026-07-14 12:51:07.512929: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_11', 4 bytes spill stores, 12 bytes spill loads



8229/8230 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - auc: 0.8287 - loss: 0.0674

2026-07-14 12:56:27.601213: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_11', 4 bytes spill stores, 12 bytes spill loads



8230/8230 ━━━━━━━━━━━━━━━━━━━━ 330s 40ms/step - auc: 0.8398 - loss: 0.0643 - val_auc: 0.7396 - val_loss: 0.0800 - learning_rate: 1.2500e-04
Epoch 2/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 325s 39ms/step - auc: 0.8534 - loss: 0.0618 - val_auc: 0.7650 - val_loss: 0.0755 - learning_rate: 1.2500e-04
Epoch 3/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 325s 39ms/step - auc: 0.8570 - loss: 0.0613 - val_auc: 0.7662 - val_loss: 0.0747 - learning_rate: 1.2500e-04
Epoch 4/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 325s 40ms/step - auc: 0.8610 - loss: 0.0607 - val_auc: 0.7656 - val_loss: 0.0790 - learning_rate: 1.2500e-04
Epoch 5/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 325s 40ms/step - auc: 0.8617 - loss: 0.0605 - val_auc: 0.7680 - val_loss: 0.0779 - learning_rate: 1.2500e-04
Epoch 6/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 326s 40ms/step - auc: 0.8639 - loss: 0.0602 - val_auc: 0.7872 - val_loss: 0.0727 - learning_rate: 1.2500e-04
Epoch 7/200
8230/8230 ━━━━━━━━━━━━━━━━━━━━ 326s 40ms/step - auc: 0.8653 - loss: 0.0599 - val_auc: 0.